In [1]:
# Base class
from src.Base.Config import Config

# Project class
from src.DopaMouv.Trajectory import TrajectoryDLC, DLCTracking
import src.utils as u

# other lib
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [2]:
path = Path("/media/filer2/T4b/Datasets/Rats/Sessions/Chrimson/NoGo/Rat226_FINcue/Cuedriven_Laser_FIN/opto_Cue_30%/opto_@Cue_S1-4p15Hz_Stop@PawLifting/Rat226_20240828/Rat226_20240828_Lefthand_Opto_Cue_S1_4p15Hz_LeftCam_CompressedVideos/Rat226_20240828_Lefthand_Opto_Cue_S1_4p15Hz_LeftCam_#1.avi")

In [3]:
path.name

'Rat226_20240828_Lefthand_Opto_Cue_S1_4p15Hz_LeftCam_#1.avi'

In [6]:
for elem in path.as_posix().split("_"): 
    print(elem)

/media/filer2/T4b/Datasets/Rats/Sessions/Chrimson/NoGo/Rat226
FINcue/Cuedriven
Laser
FIN/opto
Cue
30%/opto
@Cue
S1-4p15Hz
Stop@PawLifting/Rat226
20240828/Rat226
20240828
Lefthand
Opto
Cue
S1
4p15Hz
LeftCam
CompressedVideos/Rat226
20240828
Lefthand
Opto
Cue
S1
4p15Hz
LeftCam
#1.avi


In [37]:
def parse_filename(name: str = None) -> dict:
    import re
    
    PATTERNS = {
            "subject": r"Rat\d{3}",
            "condition": r"(Chrimson|Control|Jaw)",
            "cue_condition": r"(NoGo|NoNoGo)",
            "laser_test": r"(/Cuedriven|/NoStimLaser|/opto_LaserTest)",
            "protocol": r"opto_*_%",
            "handedness": r"(Lefthand|Righthand)",
            "pulse_type": r"\dp",
            "pulse_num": r"_Stop|FULL",
            "pulse_frec": r"\d{2}Hz",
            "camera_view": r"(RightCam|LeftCam)",
            "date": r"(20\d{6})",
            "clip": r"clip_(\d+)",
            }

    result = {key: "Unknown" for key in PATTERNS.keys()}

    # First pass: regex extraction
    for key, regex in PATTERNS.items():
        if result[key] == "Unknown"  :
            match = re.search(regex, name)
            if match:
                if key == "clip" : 
                    result[key] = match.group(1)
                    continue
                result[key] = match.group(0)

    return result

In [38]:
metadata = parse_filename(str(path))
metadata

{'subject': 'Rat226',
 'condition': 'Chrimson',
 'cue_condition': 'NoGo',
 'laser_test': '/Cuedriven',
 'protocol': 'Unknown',
 'handedness': 'Lefthand',
 'pulse_type': '4p',
 'pulse_num': '_Stop',
 'pulse_frec': '15Hz',
 'camera_view': 'LeftCam',
 'date': '20240828',
 'clip': 'Unknown'}

In [39]:
root = Path("/media/filer2/T4b/Datasets/Rats/Sessions/Chrimson/")

In [40]:
records = []
for vid_path in root.rglob("*.avi"): 
    records.append(parse_filename(str(vid_path)))
df = pd.DataFrame(records)
df

,subject,condition,cue_condition,laser_test,protocol,handedness,pulse_type,pulse_num,pulse_frec,camera_view,date,clip
0,Rat226,Chrimson,NoGo,/Cuedriven,Unknown,Lefthand,4p,FULL,15Hz,LeftCam,20240909,Unknown
1,Rat226,Chrimson,NoGo,/Cuedriven,Unknown,Lefthand,4p,FULL,15Hz,RightCam,20240909,Unknown
2,Rat226,Chrimson,NoGo,/Cuedriven,Unknown,Lefthand,4p,FULL,15Hz,LeftCam,20240910,Unknown
3,Rat226,Chrimson,NoGo,/Cuedriven,Unknown,Lefthand,4p,FULL,15Hz,RightCam,20240910,Unknown
4,Rat226,Chrimson,NoGo,/Cuedriven,Unknown,Lefthand,4p,FULL,15Hz,LeftCam,20240911,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...
2204,Unknown,Chrimson,NoNoGo,Unknown,Unknown,Righthand,4p,Unknown,15Hz,RightCam,20250506,Unknown
2205,Unknown,Chrimson,NoNoGo,Unknown,Unknown,Righthand,4p,Unknown,15Hz,RightCam,20250506,Unknown
2206,Unknown,Chrimson,NoNoGo,Unknown,Unknown,Righthand,4p,Unknown,15Hz,RightCam,20250506,Unknown
2207,Unknown,Chrimson,NoNoGo,Unknown,Unknown,Righthand,4p,Unknown,15Hz,RightCam,20250506,Unknown


In [43]:
df.columns

Index(['subject', 'condition', 'cue_condition', 'laser_test', 'protocol',
       'handedness', 'pulse_type', 'pulse_num', 'pulse_frec', 'camera_view',
       'date', 'clip'],
      dtype='object')

In [44]:
import plotly.express as px
from plotly.subplots import make_subplots

In [ ]:
groups = ['condition', 'cue_condition', 'laser_test', 'protocol',
       'handedness', 'pulse_type', 'pulse_num', 'pulse_frec', 'subject', 'camera_view',
       'date'
       ]
show_total = True

counts = (
    df
    .groupby(groups)
    .size()
    .reset_index(name="count")
)

fig = px.sunburst(
    counts,
    path=groups,
    values="count",
    color_discrete_sequence=px.colors.qualitative.D3   
)

fig.update_layout(title="Video names")

fig.update_traces(
    branchvalues="total" if show_total else "remainder",
    texttemplate="<b>%{label}</b><br>%{value} (%{percentParent:.0%})",
    textfont_size=25
)

fig.write_html("dopamouv_meta.html", auto_open=True)


In [46]:
path_list = list(root.rglob("*.avi|*.mp4"))

In [76]:
import pyMRAW

video_cfg_path = "/media/filer2/T4b/Datasets/Rats/Sessions/Chrimson/NoGo/Rat233_FINcue/Cuedriven_Laser_FIN/opto_Cue_30%/opto_@Cue_S1-4p15Hz_Stop@PawLifting/Rat233_20240829/Rat233_20240829_Lefthand_Opto_Cue_S1_4p15Hz_LeftCam_CompressedVideos/Rat233_20240829_Lefthand_Opto_Cue_S1_4p15Hz_LeftCam_#1.cihx"
print(Path(video_cfg_path).exists())
images, info = pyMRAW.load_video(video_cfg_path)

True


ValueError: Unknown format code 'g' for object of type 'str'

In [73]:
images, info = pyMRAW.load_video('./data/beam.cihx')